# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gullahmadbhatti0155/MLtask1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import os
import pandas as pd
import numpy as np

data_path = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = r'C:\Users\gulla\MLtask1\data\raw\content_refresh_anonymized.csv'

df = pd.read_csv(data_path)

imp_col = 'search_volume' if 'search_volume' in df.columns else 'impressions'
clicks_col = 'clicks' if 'clicks' in df.columns else 'competition'
pos_col = 'avg_position' if 'avg_position' in df.columns else df.columns[4] if len(df.columns) > 4 else df.columns[3]
id_col = 'content_id' if 'content_id' in df.columns else df.columns[0]

df[imp_col] = pd.to_numeric(df[imp_col], errors='coerce').fillna(0)
df[clicks_col] = pd.to_numeric(df[clicks_col], errors='coerce').fillna(0)
df[pos_col] = pd.to_numeric(df[pos_col], errors='coerce').fillna(0)
df['calculated_ctr'] = np.where(df[imp_col] > 0, (df[clicks_col] / df[imp_col]) * 100, 0.0)

def assign_archetype_and_reason(row):
    vol = row[imp_col]
    ctr = row['calculated_ctr']
    pos = row[pos_col]
    
    if vol >= 1000 and ctr < 1.5:
        return 'HIGH_VOL_LOW_CTR', 'ACTION_TITLE_TAG_OPTIMIZE', 'High impression demand with below-average CTR'
    elif pos > 10.0 and vol >= 500:
        return 'STRIKING_DISTANCE', 'ACTION_CONTENT_EXPAND', 'Ranking on page 2+ with solid volume opportunity'
    elif vol < 200 and ctr < 1.0:
        return 'DECAYING_LONG_TAIL', 'ACTION_PRUNE_OR_MERGE', 'Low traffic volume and poor engagement'
    else:
        return 'STABLE_PERFORMER', 'ACTION_MONITOR_ONLY', 'Performance within nominal limits'

res = df.apply(assign_archetype_and_reason, axis=1)
df['archetype'] = [r[0] for r in res]
df['recommended_action'] = [r[1] for r in res]
df['reason_code'] = [r[2] for r in res]

df['priority_score'] = (df[imp_col] * 0.5) + ((20 - df[pos_col].clip(upper=20)) * 25) - (df['calculated_ctr'] * 10)
ranked_queue = df.sort_values(by='priority_score', ascending=False)

print("=== RANKED ACTION QUEUE SAMPLE ===")
print(ranked_queue[[id_col, 'archetype', 'recommended_action', 'priority_score', 'reason_code']].head(10).to_string())

=== RANKED ACTION QUEUE SAMPLE ===
                 content_id         archetype         recommended_action  priority_score                                    reason_code
12140  content_ef99c4abd9ab  HIGH_VOL_LOW_CTR  ACTION_TITLE_TAG_OPTIMIZE    36999.998919  High impression demand with below-average CTR
28282  content_454cc6654c6e  HIGH_VOL_LOW_CTR  ACTION_TITLE_TAG_OPTIMIZE    30249.998182  High impression demand with below-average CTR
6972   content_bf67a444faef  HIGH_VOL_LOW_CTR  ACTION_TITLE_TAG_OPTIMIZE    30249.998182  High impression demand with below-average CTR
17907  content_5ec29ae79c60  HIGH_VOL_LOW_CTR  ACTION_TITLE_TAG_OPTIMIZE    30249.997851  High impression demand with below-average CTR
18701  content_deb54e9e19cd  HIGH_VOL_LOW_CTR  ACTION_TITLE_TAG_OPTIMIZE    30249.997851  High impression demand with below-average CTR
13502  content_f76ccf7a7834  HIGH_VOL_LOW_CTR  ACTION_TITLE_TAG_OPTIMIZE    25012.495354  High impression demand with below-average CTR
16005  conten

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use & Known Operational Limits

* **Intended Use**: This playbook provides decision-support ranking for human SEO strategists to identify high-impact content refresh opportunities across large-scale web properties.
* **Cost/Value Framework**: Prioritizes low-effort, high-return updates (e.g., meta updates, internal linking) before allocating resources to full content rewrites.
* **Operational Limits**:
  * **Static Metrics**: The model relies on batch snapshot data and does not capture real-time algorithm updates or immediate SERP volatility.
  * **Non-Linear Intent**: High impressions with low CTR may indicate brand queries or instant-answer SERP features rather than weak content titles.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human-Review Governance & No-Go Automation List

#### Human Review Rules
1. **Validation of SERP Intent**: Verify if low CTR is due to SERP layout changes (snippets, AI overviews) before rewriting titles.
2. **Technical Sanity Check**: Ensure URL drops are not caused by canonical tag misconfigurations or broken redirects.

#### No-Go List (STRICTLY PROHIBITED FROM AUTOMATION)
* **Automated Content Rewriting**: AI should never auto-publish rewritten article bodies without subject matter expert review.
* **Bulk Content Deletion/Pruning**: URL removal or 301 redirects must undergo manual approval to protect legacy backlinks.
* **Brand-Critical Pages**: High-converting landing pages or core product pages are excluded from automated action triggers.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring & Retrain Triggers

* **Data Drift Trigger**: Retrain model if median search volume or impression distributions shift by > 20% quarter-over-quarter.
* **Performance Drift Trigger**: Trigger model recalibration if the precision of recommended actions drops below 80% during human review audits.
* **Scheduled Cadence**: Re-run pipeline and update priority queues on a monthly batch execution schedule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
import matplotlib.pyplot as plt

os.makedirs('../../work/outputs', exist_ok=True)
os.makedirs('../../work/figures', exist_ok=True)

export_cols = [id_col, imp_col, clicks_col, pos_col, 'calculated_ctr', 'archetype', 'recommended_action', 'priority_score', 'reason_code']
output_csv_path = '../../work/outputs/ranked_action_queue.csv'
ranked_queue[export_cols].to_csv(output_csv_path, index=False)
print(f"Exported Ranked Queue CSV to: {output_csv_path}")

plt.figure(figsize=(8, 4))
ranked_queue['archetype'].value_counts().plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Content Archetype Distribution')
plt.xlabel('URL Count')
plt.tight_layout()

fig_path = '../../work/figures/archetype_distribution.png'
plt.savefig(fig_path, dpi=300)
plt.close()
print(f"Exported Archetype Distribution Figure to: {fig_path}")

Exported Ranked Queue CSV to: ../../work/outputs/ranked_action_queue.csv
Exported Archetype Distribution Figure to: ../../work/figures/archetype_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.